# Russia–Vietnam Trade: Scale, Structure, and Market Opportunities, 2017–2026

## TL;DR

- Bilateral trade turnover increased from **USD 3.55 billion** in 2017 to **USD 4.51 billion** in 2025, a CAGR of **3.0%** across eight intervals. Russian exports to Vietnam grew at **6.2%**, while Russian imports from Vietnam grew at **0.5%**.
- Trade turnover reached **USD 4.07 billion** in January–September 2026. This incomplete period is excluded from CAGR and is not compared with full-year 2025 as YoY because the dataset has no January–September 2025 snapshot.
- Russia's nearly balanced trade account in 2025, approximately **−USD 13 million**, hides substantial commodity asymmetry: Russia mainly supplies energy and chemicals while importing agricultural goods, textiles, and apparel.
- CAGR is also calculated for L1 and L2 and paired with absolute change, the number of growth years, and annual-growth volatility. A high CAGR from a low base is not treated as sufficient evidence of sustained growth.
- The export-opportunity matrix uses three dimensions over the comparable 2017–2025 period: Vietnam import-market CAGR, Russia's 2025 share, and 2025 market size.
- Russian exports to Vietnam became more concentrated: the effective number of L2 categories fell from **9.1** to **3.3** by 2026 YTD. Vietnamese exports to Russia diversified from **3.1** to **6.6**.
- Commodity detail is incomplete in 2017. L1/L2 changes from that base describe the structure of **published commodity rows**, without constructing an artificial residual.
- Japan is used as a benchmark for testing the hypothesis that Russia could make greater use of the geographic advantages of its Far East in trade with Vietnam.


## How to read this study

The analysis addresses four questions:

1. **How did the scale of Russia–Vietnam trade change?**
2. **What did the countries trade before, and what do they trade now?**
3. **Which products drove the change, and how did dependence on individual categories evolve?**
4. **What position does Russia hold in Vietnam's import market?**

The calculation layer remains in Pandas. All charts use Plotly and can be explored through tooltips or saved as standalone HTML/PNG files.

**Common period limitation:** 2017–2025 are completed annual snapshots, while 2026 covers January–September (`2026 YTD`). Under the adopted methodology, 2026 YTD is included in the main charts and shown alongside full years without annualization. Changes ending in 2026 therefore reflect both structural dynamics and the incomplete reporting period.


## Context and methods

### Definitions and assumptions

- `country_total` is used for trade turnover and the official balance.
- `country_commodity`, harmonized through L2, is used for structure, concentration, drivers, and commodity asymmetry.
- **HS** is the international commodity classification and is used here only as a reference attribute. **L2** is a harmonized commodity category (`commodity_id` / `canonical_name`), while **L1** is a broad analytical group.
- `Other products` is retained; `Unclassified` is excluded only from HHI, CR5, and `1/HHI`.
- The difference between `country_total` and the sum of published commodity rows is called **“Not detailed in commodity statistics.”** It is neither a balance nor a standalone commodity category and is not added to the structure.
- 2017–2025 are treated as annual snapshots. However, 2023 remains `preliminary`, while 5N for 2022 and 2025 represents November YTD snapshots labelled `final` under the adopted convention. CAGR should therefore be read as growth between the selected annual snapshots, not as a perfectly calendar-aligned rate for the export series.
- 2026 is January–September YTD. It is included in the main charts and consistently labelled as incomplete.

### CAGR method

The main comparable horizon is 2017–2025, with **eight annual intervals** between the endpoints:

\[
CAGR = \left(rac{V_{2025}}{V_{2017}}ight)^{1/8}-1
\]

CAGR is calculated only when both endpoint values are positive. Absolute change, the number of positive annual growth rates, and the standard deviation of annual growth are retained to assess stability. L2 rankings additionally require observations in all nine annual snapshots. Categories with a 2017 base below USD 1 million are excluded from the ranking but remain in the calculation table.

2026 is **excluded from CAGR**. The analytical layer has no comparable January–September 2025 snapshot: 5N for 2025 covers January–November, while 5X covers January–December. A 2026 YTD rate against the same period of 2025 is therefore not calculated; only the actual 2026 YTD level is shown.

**Russia's trade balance** is the difference between Russian exports to Vietnam and Russian imports from Vietnam:

\[
Balance_{RU}=Export_{RUightarrow VN}-Import_{RU\leftarrow VN}
\]

Source-series directions: 5N is Vietnam's imports from Russia, hence Russian exports; 5X is Vietnam's exports to Russia, hence Russian imports.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from plotly.offline import init_notebook_mode, iplot
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
init_notebook_mode(connected=False)

FLOW_LABELS = {
    "5N": "Russia's exports to Vietnam",
    "5X": "Russia's imports from Vietnam",
}
FLOW_COLORS = {"5N": "#1F5A7A", "5X": "#D89B2B"}
L1_COLORS = {
    "Agricultural and Food Products": "#2A6F97",
    "Energy Resources": "#D99B2B",
    "Chemicals, Polymers and Pharmaceuticals": "#7A8B43",
    "Metals and Mineral Raw Materials": "#B6678A",
    "Machinery, Electronics and Equipment": "#5C6B73",
    "Textiles, Apparel and Footwear": "#E07A5F",
    "Wood, Paper and Related Products": "#4F8A8B",
    "Vehicles and Transport Equipment": "#8D6A9F",
    "Construction and Ceramic Materials": "#A98467",
    "Other Manufactured Goods": "#6C757D",
    "Other Published Products": "#C5A46D",
    "Unclassified": "#BFC5CA",
}


def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root containing pyproject.toml")


project_root = find_project_root(Path.cwd().resolve())
data_dir = project_root / "data/processed/vietnam_customs"
trade_path = data_dir / "trade_facts.csv"
documents_path = data_dir / "documents.csv"
mapping_path = data_dir / "commodity_mapping_approved_2017_2026.csv"
dictionary_path = data_dir / "commodity_dictionary_approved.csv"

required_files = [trade_path, documents_path, mapping_path, dictionary_path]
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError("Required data files were not found:\n" + "\n".join(map(str, missing_files)))

output_dir = project_root / "outputs/vietnam_trade_2017_2026_extended"
figure_html_dir = output_dir / "figures_html"
figure_png_dir = output_dir / "figures_png"
output_dir.mkdir(parents=True, exist_ok=True)
figure_html_dir.mkdir(parents=True, exist_ok=True)
figure_png_dir.mkdir(parents=True, exist_ok=True)


def style_figure(fig, title, height=620, legend=True, right_margin=45, legend_orientation="h"):
    fig.update_layout(
        template="plotly_white",
        title=dict(text=title, x=0.02, xanchor="left", font=dict(size=22)),
        font=dict(family="Arial", size=13, color="#2F343A"),
        height=height,
        margin=dict(l=70, r=right_margin, t=90, b=70),
        legend=(
            dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.01)
            if legend_orientation == "v"
            else dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0)
        ),
        showlegend=legend,
        hoverlabel=dict(bgcolor="white", font_size=13),
    )
    fig.update_xaxes(showgrid=False, linecolor="#AAB0B6", zerolinecolor="#737A82")
    fig.update_yaxes(gridcolor="#E5E8EB", linecolor="#AAB0B6", zerolinecolor="#737A82")
    return fig


static_export_warnings = []


def present(fig, slug, title, height=620, legend=True, right_margin=45, legend_orientation="h"):
    style_figure(
        fig, title, height=height, legend=legend,
        right_margin=right_margin, legend_orientation=legend_orientation,
    )
    fig.write_html(figure_html_dir / f"{slug}.html", include_plotlyjs="directory", full_html=True)
    try:
        fig.write_image(figure_png_dir / f"{slug}.png", width=1600, height=height, scale=1.5)
    except Exception as exc:
        static_export_warnings.append(f"{slug}: {type(exc).__name__}: {exc}")
    iplot(fig)

print("Sources:")
print(" •", trade_path)
print(" •", documents_path)
print(" •", mapping_path)
print(" •", dictionary_path)
print("Outputs:", output_dir)

## Data preparation

Calculations are performed once; all subsequent charts use validated aggregates.


In [ ]:
facts = pd.read_csv(trade_path, low_memory=False)
documents = pd.read_csv(documents_path)
mapping = pd.read_csv(mapping_path)
dictionary = pd.read_csv(dictionary_path)
# Localize L1 labels without changing category membership or values.
dictionary["analytical_group_ru"] = dictionary["analytical_group_eng"]

required_fact_columns = {
    "series", "trade_flow", "reference_year", "reference_month", "edition",
    "fact_level", "country_name_raw", "commodity_name_raw", "ytd_value_usd",
}
assert required_fact_columns <= set(facts.columns)
assert not mapping.duplicated(["series", "commodity_name_raw"]).any()
assert dictionary["commodity_id"].is_unique

commodity_facts = facts[facts["fact_level"].isin(["country_commodity", "national_commodity"])].copy()
mapped_all = commodity_facts.merge(
    mapping, on=["series", "commodity_name_raw"], how="left", validate="many_to_one", indicator=True
)
assert mapped_all["_merge"].eq("both").all(), "Some commodity rows are not mapped"
mapped_all = mapped_all.drop(columns="_merge").merge(
    dictionary, on="commodity_id", how="left", validate="many_to_one"
)
assert mapped_all["canonical_name"].notna().all(), "Some commodity_id values are missing from the dictionary"

russia_raw = mapped_all[
    (mapped_all["fact_level"] == "country_commodity")
    & (mapped_all["country_name_raw"] == "Russian Federation")
].copy()
l2 = (
    russia_raw.groupby(
        ["reference_year", "reference_month", "edition", "series", "trade_flow",
         "commodity_id", "canonical_name", "analytical_group_ru", "analytical_group_eng",
         "analytical_group_hs", "analytical_group_hs_code"],
        dropna=False, as_index=False,
    )["ytd_value_usd"].sum()
)
l2["russia_flow"] = l2["series"].map(FLOW_LABELS)
l2["published_total_usd"] = l2.groupby(["reference_year", "series"])["ytd_value_usd"].transform("sum")
l2["published_share"] = l2["ytd_value_usd"] / l2["published_total_usd"]

l1 = (
    l2.groupby(["reference_year", "series", "russia_flow", "analytical_group_ru"], as_index=False)["ytd_value_usd"].sum()
)
l1["published_total_usd"] = l1.groupby(["reference_year", "series"])["ytd_value_usd"].transform("sum")
l1["share"] = l1["ytd_value_usd"] / l1["published_total_usd"]

japan_raw = mapped_all[
    (mapped_all["fact_level"] == "country_commodity")
    & (mapped_all["country_name_raw"] == "Japan")
].copy()
japan_l2 = (
    japan_raw.groupby(
        ["reference_year", "series", "commodity_id", "canonical_name", "analytical_group_ru"],
        dropna=False, as_index=False,
    )["ytd_value_usd"].sum()
)
japan_l1 = (
    japan_l2.groupby(["reference_year", "series", "analytical_group_ru"], as_index=False)["ytd_value_usd"].sum()
)
japan_l1["published_total_usd"] = japan_l1.groupby(
    ["reference_year", "series"]
)["ytd_value_usd"].transform("sum")
japan_l1["share"] = japan_l1["ytd_value_usd"] / japan_l1["published_total_usd"]

def period_label(year):
    return "2026 YTD" if int(year) == 2026 else str(int(year))

PERIOD_ORDER = [str(year) for year in range(2017, 2026)] + ["2026 YTD"]

russia_totals_long = facts[
    (facts["fact_level"] == "country_total")
    & (facts["country_name_raw"] == "Russian Federation")
    & facts["series"].isin(["5N", "5X"])
].copy()
trade_totals = (
    russia_totals_long.pivot(index="reference_year", columns="series", values="ytd_value_usd")
    .rename(columns={"5N": "russia_exports_usd", "5X": "russia_imports_usd"})
    .reset_index()
)
trade_totals["turnover_usd"] = trade_totals["russia_exports_usd"] + trade_totals["russia_imports_usd"]
trade_totals["russia_trade_balance_usd"] = trade_totals["russia_exports_usd"] - trade_totals["russia_imports_usd"]

raw_sums = russia_raw.groupby(["reference_year", "series"])["ytd_value_usd"].sum().rename("raw_sum")
l2_sums = l2.groupby(["reference_year", "series"])["ytd_value_usd"].sum().rename("published_commodity_sum_usd")
aggregation_check = pd.concat([raw_sums, l2_sums], axis=1).reset_index()
aggregation_check["difference_usd"] = aggregation_check["published_commodity_sum_usd"] - aggregation_check["raw_sum"]
assert aggregation_check["difference_usd"].abs().max() == 0

reconciliation = (
    russia_totals_long[["reference_year", "reference_month", "series", "edition", "ytd_value_usd"]]
    .rename(columns={"ytd_value_usd": "country_total_usd"})
    .merge(l2_sums.reset_index(), on=["reference_year", "series"], how="left", validate="one_to_one")
)
reconciliation["not_detailed_usd"] = reconciliation["country_total_usd"] - reconciliation["published_commodity_sum_usd"]
reconciliation["coverage_pct"] = 100 * reconciliation["published_commodity_sum_usd"] / reconciliation["country_total_usd"]
reconciliation["russia_flow"] = reconciliation["series"].map(FLOW_LABELS)

assert len(russia_raw) == 466
assert len(mapped_all) == 23_499
print(f"Ready: {len(mapped_all):,} commodity rows mapped; L2 aggregation preserves totals exactly.")

## 1. How did the scale of trade change?


In [ ]:
annual_totals = trade_totals[trade_totals["reference_year"] <= 2026].copy()
annual_totals["period"] = annual_totals["reference_year"].map(period_label)
long_annual = annual_totals.melt(
    id_vars=["reference_year", "period"],
    value_vars=["russia_exports_usd", "russia_imports_usd", "turnover_usd"],
    var_name="metric", value_name="value_usd",
)
long_annual["metric_label"] = long_annual["metric"].map({
    "russia_exports_usd": "Russia's exports to Vietnam",
    "russia_imports_usd": "Russia's imports from Vietnam",
    "turnover_usd": "Trade turnover",
})
metric_colors = {
    "Russia's exports to Vietnam": FLOW_COLORS["5N"],
    "Russia's imports from Vietnam": FLOW_COLORS["5X"],
    "Trade turnover": "#353B46",
}

fig = make_subplots(rows=1, cols=2, column_widths=[0.68, 0.32], horizontal_spacing=0.12,
                    subplot_titles=("Trade flows", "Russia's trade balance"))
for label in metric_colors:
    part = long_annual[long_annual["metric_label"] == label]
    fig.add_trace(go.Scatter(
        x=part["period"], y=part["value_usd"] / 1e9,
        mode="lines+markers", name=label, line=dict(color=metric_colors[label], width=3),
        marker=dict(size=8), hovertemplate="%{x}: $%{y:.2f} billion<extra>" + label + "</extra>",
    ), row=1, col=1)

balance_colors = np.where(annual_totals["russia_trade_balance_usd"] >= 0, "#1F5A7A", "#D89B2B")
fig.add_trace(go.Bar(
    x=annual_totals["period"], y=annual_totals["russia_trade_balance_usd"] / 1e9,
    marker_color=balance_colors, showlegend=False,
    customdata=annual_totals["russia_trade_balance_usd"] / 1e6,
    hovertemplate="%{x}: %{customdata:+.0f} USD million<extra></extra>",
), row=1, col=2)
fig.add_hline(y=0, line_color="#555D65", line_width=1, row=1, col=2)
fig.update_xaxes(title_text="Period", categoryorder="array", categoryarray=PERIOD_ORDER)
fig.update_yaxes(title_text="USD billion", row=1, col=1)
fig.update_yaxes(title_text="USD billion", row=1, col=2)
present(fig, "01_trade_scale", "Scale of bilateral trade, 2017–2026 YTD", height=620)

completed = annual_totals[annual_totals["reference_year"] <= 2025].copy()
cagr = lambda column: 100 * ((completed[column].iloc[-1] / completed[column].iloc[0]) ** (1 / 8) - 1)
ytd = annual_totals[annual_totals["reference_year"] == 2026].iloc[0]
display(Markdown(
    f"Trade turnover increased from **${completed.turnover_usd.iloc[0]/1e9:.2f} billion** in 2017 to "
    f"**${completed.turnover_usd.iloc[-1]/1e9:.2f} billion** in 2025 Full-year CAGR — **{cagr('turnover_usd'):.1f}%**. "
    f"Trade turnover in January–September 2026 was **${ytd.turnover_usd/1e9:.2f} billion**. "
    f"Russia's trade balance in 2025 was "
    f"**{completed.russia_trade_balance_usd.iloc[-1]/1e6:+.0f} USD million**"
))

### 1.1. CAGR of total trade, 2017–2025

CAGR answers what constant annual growth rate would connect the initial and final snapshots. It does not describe the path between them, so absolute change and annual dynamics are shown alongside it. 2026 YTD is excluded from this calculation.


In [ ]:
CAGR_START_YEAR = 2017
CAGR_END_YEAR = 2025
CAGR_INTERVALS = CAGR_END_YEAR - CAGR_START_YEAR
CAGR_YEARS = list(range(CAGR_START_YEAR, CAGR_END_YEAR + 1))


def calculate_cagr(start_value, end_value, intervals=CAGR_INTERVALS):
    if pd.isna(start_value) or pd.isna(end_value) or start_value <= 0 or end_value <= 0:
        return np.nan
    return (end_value / start_value) ** (1 / intervals) - 1


metric_names = {
    "russia_exports_usd": "Russia's exports to Vietnam",
    "russia_imports_usd": "Russia's imports from Vietnam",
    "turnover_usd": "Trade turnover",
}
start_total = trade_totals.loc[trade_totals["reference_year"] == CAGR_START_YEAR].iloc[0]
end_total = trade_totals.loc[trade_totals["reference_year"] == CAGR_END_YEAR].iloc[0]
total_cagr_summary = pd.DataFrame([
    {
        "metric": label,
        "value_2017_usd": start_total[column],
        "value_2025_usd": end_total[column],
        "absolute_change_usd": end_total[column] - start_total[column],
        "cagr_pct": 100 * calculate_cagr(start_total[column], end_total[column]),
    }
    for column, label in metric_names.items()
])

fig = go.Figure(go.Bar(
    x=total_cagr_summary["cagr_pct"],
    y=total_cagr_summary["metric"],
    orientation="h",
    marker_color=[FLOW_COLORS["5N"], FLOW_COLORS["5X"], "#353B46"],
    customdata=np.c_[
        total_cagr_summary["value_2017_usd"] / 1e9,
        total_cagr_summary["value_2025_usd"] / 1e9,
        total_cagr_summary["absolute_change_usd"] / 1e9,
    ],
    text=total_cagr_summary["cagr_pct"].map(lambda value: f"{value:.1f}%"),
    textposition="outside",
    hovertemplate=(
        "%{y}<br>CAGR: %{x:.1f}%"
        "<br>2017: $%{customdata[0]:.2f} billion"
        "<br>2025: $%{customdata[1]:.2f} billion"
        "<br>Absolute change: $%{customdata[2]:+.2f} billion<extra></extra>"
    ),
))
fig.add_vline(x=0, line_color="#555D65", line_width=1)
fig.update_xaxes(title_text="CAGR, % per year")
present(fig, "01b_total_trade_cagr", "CAGR of bilateral trade, 2017–2025", height=440, legend=False)

display(total_cagr_summary.assign(
    **{
        "2017, USD billion": total_cagr_summary["value_2017_usd"] / 1e9,
        "2025, USD billion": total_cagr_summary["value_2025_usd"] / 1e9,
        "Absolute change, USD billion": total_cagr_summary["absolute_change_usd"] / 1e9,
        "CAGR, %": total_cagr_summary["cagr_pct"],
    }
)[["metric", "2017, USD billion", "2025, USD billion", "Absolute change, USD billion", "CAGR, %"]]
.rename(columns={"metric": "Metric"}).round(2))

## 2. What did the countries trade before, and what do they trade now?

### 2.1. Evolution of the L1 structure


In [ ]:
annual_l1 = l1[l1["reference_year"] <= 2026].copy()
annual_l1["period"] = annual_l1["reference_year"].map(period_label)
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                    subplot_titles=(FLOW_LABELS["5N"], FLOW_LABELS["5X"]))
l1_order = (
    annual_l1.groupby("analytical_group_ru")["ytd_value_usd"].sum().sort_values(ascending=False).index.tolist()
)
for col, series in enumerate(["5N", "5X"], start=1):
    flow = annual_l1[annual_l1["series"] == series]
    for group in l1_order:
        part = flow[flow["analytical_group_ru"] == group]
        if part.empty:
            continue
        fig.add_trace(go.Scatter(
            x=part["period"], y=part["share"] * 100,
            mode="lines", stackgroup=f"stack-{series}", groupnorm="percent",
            name=group, legendgroup=group, showlegend=(col == 1),
            line=dict(color=L1_COLORS[group], width=0.7),
            hovertemplate="%{x}: %{y:.1f}%<extra>" + group + "</extra>",
        ), row=1, col=col)
fig.update_xaxes(title_text="Period", categoryorder="array", categoryarray=PERIOD_ORDER)
fig.update_yaxes(title_text="Share of published commodity value, %", range=[0, 100], row=1, col=1)
fig.update_yaxes(title_text="", range=[0, 100], row=1, col=2)
present(fig, "02_l1_stacked_area", "Evolution of the L1 trade structure, 2017–2026 YTD",
        height=700, right_margin=330, legend_orientation="v")

### 2.2. Change in the L1 structure: 2017 → 2026 YTD


In [ ]:
def l1_dumbbell(series):
    wide = (
        annual_l1[(annual_l1["series"] == series) & annual_l1["reference_year"].isin([2017, 2026])]
        .pivot(index="analytical_group_ru", columns="reference_year", values="share")
        .fillna(0)
        .reset_index()
    )
    wide["change_pp"] = 100 * (wide[2026] - wide[2017])
    wide = wide.sort_values("change_pp")
    fig = go.Figure()
    for _, row in wide.iterrows():
        fig.add_trace(go.Scatter(
            x=[100 * row[2017], 100 * row[2026]], y=[row["analytical_group_ru"]] * 2,
            mode="lines", line=dict(color="#B8BEC4", width=3), showlegend=False,
            hoverinfo="skip",
        ))
    fig.add_trace(go.Scatter(
        x=100 * wide[2017], y=wide["analytical_group_ru"], mode="markers",
        name="2017", marker=dict(color="#AAB0B6", size=11, symbol="circle-open", line_width=2),
        customdata=wide["change_pp"],
        hovertemplate="2017: %{x:.1f}%<br>Change: %{customdata:+.1f} pp<extra>%{y}</extra>",
    ))
    fig.add_trace(go.Scatter(
        x=100 * wide[2026], y=wide["analytical_group_ru"], mode="markers+text",
        name="2026 YTD", marker=dict(color=FLOW_COLORS[series], size=12),
        text=[f"{v:+.1f} pp" for v in wide["change_pp"]], textposition="middle right",
        customdata=wide["change_pp"],
        hovertemplate="2026 YTD: %{x:.1f}%<br>Change: %{customdata:+.1f} pp<extra>%{y}</extra>",
    ))
    fig.update_xaxes(title_text="Share of published commodity value, %")
    return fig

for series, slug in [("5N", "03a_l1_dumbbell_exports"), ("5X", "03b_l1_dumbbell_imports")]:
    present(l1_dumbbell(series), slug, f"L1: {FLOW_LABELS[series]}, 2017 → 2026 YTD", height=650)

### 2.3. Absolute dynamics of the largest L1 groups

Using a common scale within each trade direction separates value change from share change.


In [ ]:
def l1_small_multiples(series):
    flow = annual_l1[annual_l1["series"] == series].copy()
    top_groups = (
        flow.groupby("analytical_group_ru")["ytd_value_usd"].sum().nlargest(6).index.tolist()
    )
    plot_data = flow[flow["analytical_group_ru"].isin(top_groups)].copy()
    plot_data["value_usd_m"] = plot_data["ytd_value_usd"] / 1e6
    plot_data["period"] = plot_data["reference_year"].map(period_label)
    fig = px.line(
        plot_data, x="period", y="value_usd_m", color="analytical_group_ru",
        facet_col="analytical_group_ru", facet_col_wrap=3, markers=True,
        color_discrete_map=L1_COLORS, category_orders={"analytical_group_ru": top_groups},
        labels={"period": "Period", "value_usd_m": "USD million ", "analytical_group_ru": "L1"},
    )
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig.update_yaxes(matches="y", rangemode="tozero")
    fig.update_xaxes(categoryorder="array", categoryarray=PERIOD_ORDER)
    fig.update_layout(showlegend=False)
    return fig

present(l1_small_multiples("5N"), "04a_l1_absolute_exports",
        "Absolute dynamics of the largest L1 groups: Russian exports", height=760, legend=False)
present(l1_small_multiples("5X"), "04b_l1_absolute_imports",
        "Absolute dynamics of the largest L1 groups: Russian imports", height=760, legend=False)

### 2.4. L1 CAGR and trajectory stability, 2017–2025

Bars show the average annual growth rate. Tooltips retain the initial and final values, absolute change, the number of growth years out of eight, and annual-growth volatility. “Sustained growth” means positive CAGR and at least six growth years; “sustained decline” means negative CAGR and at least six decline years. This is an analytical rule, not a statistical test.


In [ ]:
def build_cagr_metrics(data, index_columns, value_column="ytd_value_usd", low_base_usd=0):
    wide = data[data["reference_year"].between(CAGR_START_YEAR, CAGR_END_YEAR)].pivot_table(
        index=index_columns, columns="reference_year", values=value_column, aggfunc="sum"
    ).reindex(columns=CAGR_YEARS)
    annual_values = wide[CAGR_YEARS]
    annual_yoy = annual_values.pct_change(axis=1, fill_method=None)

    result = wide.reset_index()
    result["observed_years"] = annual_values.notna().sum(axis=1).to_numpy()
    result["value_2017_usd"] = annual_values[CAGR_START_YEAR].to_numpy()
    result["value_2025_usd"] = annual_values[CAGR_END_YEAR].to_numpy()
    result["absolute_change_usd"] = result["value_2025_usd"] - result["value_2017_usd"]
    result["cagr_pct"] = [
        100 * calculate_cagr(start, end)
        for start, end in zip(result["value_2017_usd"], result["value_2025_usd"])
    ]
    result["growth_years"] = (annual_yoy > 0).sum(axis=1).to_numpy()
    result["decline_years"] = (annual_yoy < 0).sum(axis=1).to_numpy()
    result["yoy_volatility_pp"] = (100 * annual_yoy.std(axis=1)).to_numpy()
    result["complete_period"] = result["observed_years"].eq(len(CAGR_YEARS))
    result["low_base"] = result["value_2017_usd"].lt(low_base_usd)
    result["trajectory"] = np.select(
        [
            result["low_base"] & result["cagr_pct"].gt(0),
            result["complete_period"] & result["cagr_pct"].gt(0) & result["growth_years"].ge(6),
            result["complete_period"] & result["cagr_pct"].lt(0) & result["decline_years"].ge(6),
            result["cagr_pct"].gt(0),
            result["cagr_pct"].lt(0),
        ],
        [
            "Growth from a low base", "Sustained growth", "Sustained decline",
            "Volatile growth", "Volatile decline",
        ],
        default="Insufficient data",
    )
    return result


l1_cagr = build_cagr_metrics(
    l1,
    ["series", "russia_flow", "analytical_group_ru"],
    low_base_usd=10_000_000,
)
l1_cagr_plot = l1_cagr[l1_cagr["complete_period"] & l1_cagr["cagr_pct"].notna()].copy()

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.26,
    subplot_titles=(FLOW_LABELS["5N"], FLOW_LABELS["5X"]),
)
for col, series in enumerate(["5N", "5X"], start=1):
    part = l1_cagr_plot[l1_cagr_plot["series"] == series].sort_values("cagr_pct")
    colors = np.where(part["cagr_pct"] >= 0, FLOW_COLORS["5N"], FLOW_COLORS["5X"])
    fig.add_trace(go.Bar(
        x=part["cagr_pct"], y=part["analytical_group_ru"], orientation="h",
        marker_color=colors, showlegend=False,
        customdata=np.c_[
            part["value_2017_usd"] / 1e6,
            part["value_2025_usd"] / 1e6,
            part["absolute_change_usd"] / 1e6,
            part["growth_years"],
            part["yoy_volatility_pp"],
            part["trajectory"],
        ],
        hovertemplate=(
            "%{y}<br>CAGR: %{x:.1f}%"
            "<br>2017: $%{customdata[0]:.1f} million; 2025: $%{customdata[1]:.1f} million"
            "<br>Absolute change: $%{customdata[2]:+.1f} million"
            "<br>Growth years: %{customdata[3]:.0f} out of 8"
            "<br>YoY volatility: %{customdata[4]:.1f} pp"
            "<br>%{customdata[5]}<extra></extra>"
        ),
    ), row=1, col=col)
    fig.add_vline(x=0, line_color="#555D65", line_width=1, row=1, col=col)
fig.update_xaxes(title_text="CAGR, % per year")
present(fig, "04c_l1_cagr", "L1: CAGR and trajectory stability, 2017–2025", height=680, legend=False)

display(l1_cagr_plot[[
    "russia_flow", "analytical_group_ru", "value_2017_usd", "value_2025_usd",
    "absolute_change_usd", "cagr_pct", "growth_years", "yoy_volatility_pp", "trajectory",
]].rename(columns={
    "russia_flow": "Direction", "analytical_group_ru": "L1",
    "value_2017_usd": "2017, USD", "value_2025_usd": "2025, USD",
    "absolute_change_usd": "Absolute change, USD", "cagr_pct": "CAGR, %",
    "growth_years": "Growth years out of 8", "yoy_volatility_pp": "YoY volatility, pp",
    "trajectory": "Trajectory",
}).sort_values(["Direction", "CAGR, %"], ascending=[True, False]).round(1))

### 2.5. Russia and Japan: an L1 benchmark

Solid lines show the structure of Russia's trade with Vietnam; dotted lines show Japan's. The comparison does not assume equal logistics costs. It is a benchmark for the hypothesis that Russia could use the geographic opportunities of its Far East more effectively. Any potential redistribution of supply between Russia and Japan remains a hypothesis requiring evidence on routes, port infrastructure, and transport costs.


In [ ]:
def compare_with_japan(series):
    russia = annual_l1[["reference_year", "series", "analytical_group_ru", "share"]].copy()
    russia["country"] = "Russia"
    japan = japan_l1.copy()
    japan["country"] = "Japan"
    combined = pd.concat([russia, japan], ignore_index=True)
    combined = combined[combined["series"] == series].copy()
    top_groups = (
        russia[(russia["series"] == series) & (russia["reference_year"] == 2026)]
        .nlargest(6, "share")["analytical_group_ru"].tolist()
    )
    combined = combined[combined["analytical_group_ru"].isin(top_groups)].copy()
    combined["share_pct"] = 100 * combined["share"]
    combined["period"] = combined["reference_year"].map(period_label)
    fig = px.line(
        combined, x="period", y="share_pct", color="country", line_dash="country",
        facet_col="analytical_group_ru", facet_col_wrap=3,
        color_discrete_map={"Russia": FLOW_COLORS[series], "Japan": "#6F7780"},
        line_dash_map={"Russia": "solid", "Japan": "dot"},
        category_orders={"period": PERIOD_ORDER, "analytical_group_ru": top_groups},
        labels={"period": "Period", "share_pct": "Share of published value, %", "country": ""},
    )
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig.update_yaxes(matches="y", rangemode="tozero")
    fig.update_traces(line_width=3)
    return fig

present(compare_with_japan("5N"), "05a_l1_russia_japan_imports_vietnam",
        "Vietnam imports from Russia and Japan: L1 structure", height=760,
        right_margin=170, legend_orientation="v")
present(compare_with_japan("5X"), "05b_l1_russia_japan_exports_vietnam",
        "Vietnam exports to Russia and Japan: L1 structure", height=760,
        right_margin=170, legend_orientation="v")

### 2.6. Structural asymmetry of bilateral trade


In [ ]:
l1_current = l1[l1["reference_year"] == 2026].pivot_table(
    index="analytical_group_ru", columns="series", values="ytd_value_usd", fill_value=0
).reset_index()
l1_current["l1_trade_balance_usd"] = l1_current.get("5N", 0) - l1_current.get("5X", 0)
l1_current = l1_current.sort_values("l1_trade_balance_usd")
l1_current["value_usd_m"] = l1_current["l1_trade_balance_usd"] / 1e6
l1_current["side"] = np.where(l1_current["l1_trade_balance_usd"] >= 0,
                            "Russian exports dominate", "Russian imports dominate")
fig = px.bar(
    l1_current, x="value_usd_m", y="analytical_group_ru", orientation="h", color="side",
    color_discrete_map={"Russian exports dominate": FLOW_COLORS["5N"],
                        "Russian imports dominate": FLOW_COLORS["5X"]},
    labels={"value_usd_m": "Exports minus imports, USD million", "analytical_group_ru": "L1", "side": ""},
)
fig.update_traces(texttemplate="%{x:+.0f}", textposition="outside",
                  hovertemplate="%{y}<br>Category balance: %{x:+.0f} USD million<extra></extra>")
fig.add_vline(x=0, line_color="#555D65", line_width=1)
present(fig, "06_l1_trade_asymmetry", "L1 trade asymmetry, 2026 YTD", height=650)

published_balance = l1_current["l1_trade_balance_usd"].sum()
official_balance = annual_totals.loc[annual_totals["reference_year"] == 2026, "russia_trade_balance_usd"].iloc[0]
display(Markdown(
    f"Sum of category balances in published rows: **{published_balance/1e6:+.0f} USD million**; "
    f"official total balance: **{official_balance/1e6:+.0f} USD million**. "
    "The difference reflects the undisaggregated part of the publication, not a separate commodity category."
))

## 3. Retrospective analysis: which products changed bilateral trade?

**How to read L2:** each category combines comparable source commodity rows across statistical releases. This supports analysis over time without presenting an analytical grouping as official HS detail.

### 3.1. Largest L2 categories in 2026 YTD


In [ ]:
l2_current = l2[(l2["reference_year"] == 2026) & (l2["commodity_id"] != "C074")].copy()
l2_current["value_usd_m"] = l2_current["ytd_value_usd"] / 1e6

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.22,
                    subplot_titles=(FLOW_LABELS["5N"], FLOW_LABELS["5X"]))
for col, series in enumerate(["5N", "5X"], start=1):
    part = l2_current[l2_current["series"] == series].nlargest(10, "ytd_value_usd").sort_values("ytd_value_usd")
    fig.add_trace(go.Bar(
        x=part["value_usd_m"], y=part["canonical_name"], orientation="h",
        marker_color=FLOW_COLORS[series], showlegend=False,
        customdata=100 * part["published_share"],
        hovertemplate="%{y}<br>$%{x:.1f} million<br>%{customdata:.1f}% of published value<extra></extra>",
    ), row=1, col=col)
fig.update_xaxes(title_text="Value, USD million")
present(fig, "07_l2_top10_2026_ytd", "Largest published L2 categories, 2026 YTD", height=720, legend=False)

### 3.2. L2 contribution to value change, 2017 → 2026 YTD


In [ ]:
l2_period = l2[(l2["reference_year"] <= 2026) & (l2["commodity_id"] != "C074")].copy()
l2_change = (
    l2_period[l2_period["reference_year"].isin([2017, 2026])]
    .pivot_table(index=["series", "russia_flow", "commodity_id", "canonical_name", "analytical_group_ru"],
                 columns="reference_year", values="ytd_value_usd", fill_value=0)
    .reset_index()
)
l2_change["value_2017_usd"] = l2_change.get(2017, 0)
l2_change["value_2026_usd"] = l2_change.get(2026, 0)
l2_change["delta_usd"] = l2_change["value_2026_usd"] - l2_change["value_2017_usd"]
l2_change["delta_usd_m"] = l2_change["delta_usd"] / 1e6

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.20,
                    subplot_titles=(FLOW_LABELS["5N"], FLOW_LABELS["5X"]))
for col, series in enumerate(["5N", "5X"], start=1):
    part = l2_change[l2_change["series"] == series]
    selected = pd.concat([part.nlargest(7, "delta_usd"), part.nsmallest(7, "delta_usd")]).drop_duplicates()
    selected = selected.sort_values("delta_usd")
    colors = np.where(selected["delta_usd"] >= 0, FLOW_COLORS["5N"], FLOW_COLORS["5X"])
    fig.add_trace(go.Bar(
        x=selected["delta_usd_m"], y=selected["canonical_name"], orientation="h",
        marker_color=colors, showlegend=False,
        customdata=np.c_[selected["value_2017_usd"] / 1e6, selected["value_2026_usd"] / 1e6],
        hovertemplate="%{y}<br>2017: $%{customdata[0]:.1f} million<br>2026 YTD: $%{customdata[1]:.1f} million"
                      "<br>Change: %{x:+.1f} million<extra></extra>",
    ), row=1, col=col)
    fig.add_vline(x=0, line_color="#555D65", line_width=1, row=1, col=col)
fig.update_xaxes(title_text="Value change, USD million")
present(fig, "08_l2_contribution", "L2 categories driving the change in trade value", height=760, legend=False)
display(Markdown(
    "This is an **accounting contribution to value change**, not evidence of a causal mechanism. "
    "Incomplete 2017 detail particularly affects the interpretation of `Other products` and categories absent from that publication."
))

### 3.3. L2 CAGR and trajectory stability, 2017–2025

The ranking uses categories published in all nine annual snapshots and worth at least USD 1 million in 2017. This reduces distortions from missing observations and very low bases. Absolute change and volatility remain in the tooltip and final table; CAGR does not replace them.


In [ ]:
l2_cagr = build_cagr_metrics(
    l2[l2["commodity_id"] != "C074"],
    ["series", "russia_flow", "commodity_id", "canonical_name", "analytical_group_ru"],
    low_base_usd=1_000_000,
)
l2_cagr_eligible = l2_cagr[
    l2_cagr["complete_period"]
    & l2_cagr["cagr_pct"].notna()
    & ~l2_cagr["low_base"]
].copy()

selected_l2_cagr = []
for series in ["5N", "5X"]:
    part = l2_cagr_eligible[l2_cagr_eligible["series"] == series]
    selected_l2_cagr.append(pd.concat([
        part.nlargest(6, "cagr_pct"),
        part.nsmallest(6, "cagr_pct"),
    ]).drop_duplicates(["series", "commodity_id"]))
selected_l2_cagr = pd.concat(selected_l2_cagr, ignore_index=True)

fig = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.22,
    subplot_titles=(FLOW_LABELS["5N"], FLOW_LABELS["5X"]),
)
for col, series in enumerate(["5N", "5X"], start=1):
    part = selected_l2_cagr[selected_l2_cagr["series"] == series].sort_values("cagr_pct")
    colors = np.where(part["cagr_pct"] >= 0, FLOW_COLORS["5N"], FLOW_COLORS["5X"])
    fig.add_trace(go.Bar(
        x=part["cagr_pct"], y=part["canonical_name"], orientation="h",
        marker_color=colors, showlegend=False,
        customdata=np.c_[
            part["analytical_group_ru"],
            part["value_2017_usd"] / 1e6,
            part["value_2025_usd"] / 1e6,
            part["absolute_change_usd"] / 1e6,
            part["growth_years"],
            part["yoy_volatility_pp"],
            part["trajectory"],
        ],
        hovertemplate=(
            "%{y}<br>L1: %{customdata[0]}<br>CAGR: %{x:.1f}%"
            "<br>2017: $%{customdata[1]:.1f} million; 2025: $%{customdata[2]:.1f} million"
            "<br>Absolute change: $%{customdata[3]:+.1f} million"
            "<br>Growth years: %{customdata[4]:.0f} out of 8"
            "<br>YoY volatility: %{customdata[5]:.1f} pp"
            "<br>%{customdata[6]}<extra></extra>"
        ),
    ), row=1, col=col)
    fig.add_vline(x=0, line_color="#555D65", line_width=1, row=1, col=col)
fig.update_xaxes(title_text="CAGR, % per year")
present(fig, "08b_l2_cagr", "L2: categories with the highest and lowest CAGR, 2017–2025", height=820, legend=False)

display(selected_l2_cagr[[
    "russia_flow", "canonical_name", "analytical_group_ru", "value_2017_usd", "value_2025_usd",
    "absolute_change_usd", "cagr_pct", "growth_years", "yoy_volatility_pp", "trajectory",
]].rename(columns={
    "russia_flow": "Direction", "canonical_name": "L2", "analytical_group_ru": "L1",
    "value_2017_usd": "2017, USD", "value_2025_usd": "2025, USD",
    "absolute_change_usd": "Absolute change, USD", "cagr_pct": "CAGR, %",
    "growth_years": "Growth years out of 8", "yoy_volatility_pp": "YoY volatility, pp",
    "trajectory": "Trajectory",
}).sort_values(["Direction", "CAGR, %"], ascending=[True, False]).round(1))

### 3.4. L2 strategy matrix: value × share

**How to read the matrix:** the horizontal axis shows the change in monetary value, while the vertical axis shows the change in the category's share of the relevant bilateral flow. Bubble size represents value in 2026 YTD. Four quadrants distinguish strengthening positions, growth with declining share, contraction with rising share, and weakening positions. The matrix preserves absolute value dynamics; CAGR is analyzed separately and does not replace monetary contribution.


In [ ]:
shares_wide = (
    l2_period[l2_period["reference_year"].isin([2017, 2026])]
    .pivot_table(index=["series", "commodity_id", "canonical_name", "analytical_group_ru"],
                 columns="reference_year", values="published_share", fill_value=0)
    .reset_index()
)
shares_wide["share_2017"] = shares_wide.get(2017, 0)
shares_wide["share_2026"] = shares_wide.get(2026, 0)
shares_wide["share_change_pp"] = 100 * (shares_wide["share_2026"] - shares_wide["share_2017"])
l2_matrix = l2_change.merge(
    shares_wide[["series", "commodity_id", "share_2017", "share_2026", "share_change_pp"]],
    on=["series", "commodity_id"], how="left", validate="one_to_one"
)
l2_matrix["value_2026_usd_m"] = l2_matrix["value_2026_usd"] / 1e6
l2_matrix["strategy_group"] = np.select(
    [
        (l2_matrix["delta_usd"] >= 0) & (l2_matrix["share_change_pp"] >= 0),
        (l2_matrix["delta_usd"] >= 0) & (l2_matrix["share_change_pp"] < 0),
        (l2_matrix["delta_usd"] < 0) & (l2_matrix["share_change_pp"] >= 0),
    ],
    ["Strengthening position", "Growth with declining share", "Contraction with rising share"],
    default="Weakening position",
)
STRATEGY_COLORS = {
    "Strengthening position": "#1F5A7A",
    "Growth with declining share": "#D89B2B",
    "Contraction with rising share": "#7A8B43",
    "Weakening position": "#8A6A78",
}

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.10,
                    subplot_titles=(FLOW_LABELS["5N"], FLOW_LABELS["5X"]))
shown_strategy_groups = set()
for col, series in enumerate(["5N", "5X"], start=1):
    part = l2_matrix[l2_matrix["series"] == series].copy()
    sizeref = 2 * part["value_2026_usd_m"].max() / (45 ** 2)
    for group, group_data in part.groupby("strategy_group"):
        fig.add_trace(go.Scatter(
            x=group_data["delta_usd_m"], y=group_data["share_change_pp"], mode="markers",
            name=group, legendgroup=group, showlegend=(group not in shown_strategy_groups),
            marker=dict(size=group_data["value_2026_usd_m"], sizemode="area", sizeref=sizeref,
                        sizemin=5, color=STRATEGY_COLORS[group], opacity=0.78,
                        line=dict(color="white", width=1)),
            customdata=np.c_[group_data["canonical_name"], group_data["analytical_group_ru"],
                             group_data["value_2026_usd_m"], group_data["share_2017"] * 100,
                             group_data["share_2026"] * 100],
            hovertemplate="%{customdata[0]}<br>Value change: %{x:+.1f} USD million"
                          "<br>L1: %{customdata[1]}<br>Share change: %{y:+.1f} pp"
                          "<br>2026 YTD value: %{customdata[2]:.1f} USD million"
                          "<br>Share: %{customdata[3]:.1f}% → %{customdata[4]:.1f}%<extra></extra>",
        ), row=1, col=col)
        shown_strategy_groups.add(group)
    fig.add_hline(y=0, line_color="#7B8289", line_dash="dot", row=1, col=col)
    fig.add_vline(x=0, line_color="#7B8289", line_dash="dot", row=1, col=col)
fig.update_xaxes(title_text="Value change, USD million")
fig.update_yaxes(title_text="Share change, pp", row=1, col=1)
present(fig, "09_l2_strategy_matrix", "L2: value growth and changing category role, 2017 → 2026 YTD",
        height=760, right_margin=300, legend_orientation="v")

### 3.5. Diversification and monetary scale of bilateral trade

**How to read the chart:** the line shows the effective number of L2 categories, an HHI-based measure that accounts for both the category count and the value distribution. A higher line means a more even trade structure. Semi-transparent bars show the official `country_total` value of the flow.

**HHI** is the sum of squared category shares: the higher it is, the greater the concentration. Its inverse, `1/HHI`, converts the index into an intuitive effective number of equal-sized categories. **CR5** is the combined share of the five largest L2 categories and is retained in the supporting table.


In [ ]:
concentration_base = l2_period.copy()
concentration_base["hhi_share"] = concentration_base["ytd_value_usd"] / concentration_base.groupby(
    ["reference_year", "series"]
)["ytd_value_usd"].transform("sum")
concentration = (
    concentration_base.groupby(["reference_year", "series", "russia_flow"], as_index=False)
    .agg(
        hhi_fraction=("hhi_share", lambda s: np.square(s).sum()),
        cr5=("hhi_share", lambda s: s.nlargest(5).sum()),
        included_l2=("commodity_id", "nunique"),
    )
)
concentration["hhi_10000"] = concentration["hhi_fraction"] * 10_000
concentration["effective_categories"] = 1 / concentration["hhi_fraction"]
flow_values = russia_totals_long[["reference_year", "series", "ytd_value_usd"]].rename(
    columns={"ytd_value_usd": "country_total_usd"}
)
concentration = concentration.merge(flow_values, on=["reference_year", "series"], validate="one_to_one")
concentration["period"] = concentration["reference_year"].map(period_label)
concentration["country_total_usd_bn"] = concentration["country_total_usd"] / 1e9

fig = make_subplots(
    rows=1, cols=2, specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    horizontal_spacing=0.12, subplot_titles=(FLOW_LABELS["5N"], FLOW_LABELS["5X"]),
)
for col, series in enumerate(["5N", "5X"], start=1):
    part = concentration[concentration["series"] == series]
    fig.add_trace(go.Bar(
        x=part["period"], y=part["country_total_usd_bn"], name="Trade value",
        marker_color=FLOW_COLORS[series], opacity=0.25, legendgroup="value", showlegend=(col == 1),
        hovertemplate="%{x}<br>Value: $%{y:.2f} billion<extra></extra>",
    ), row=1, col=col, secondary_y=True)
    fig.add_trace(go.Scatter(
        x=part["period"], y=part["effective_categories"], name="Effective number of L2 categories",
        mode="lines+markers", line=dict(color=FLOW_COLORS[series], width=3), marker=dict(size=9),
        legendgroup="effective", showlegend=(col == 1),
        customdata=np.c_[part["hhi_10000"], part["cr5"]],
        hovertemplate="%{x}<br>Effective number of L2 categories: %{y:.1f}<br>HHI: %{customdata[0]:.0f}"
                      "<br>CR5: %{customdata[1]:.1%}<extra></extra>",
    ), row=1, col=col, secondary_y=False)
    fig.update_yaxes(title_text="Effective number of L2 categories", row=1, col=col, secondary_y=False)
    fig.update_yaxes(title_text="Value, USD billion",
                     range=[0, part["country_total_usd_bn"].max() * 1.12],
                     row=1, col=col, secondary_y=True)
fig.update_xaxes(title_text="Period", categoryorder="array", categoryarray=PERIOD_ORDER)
present(fig, "10_diversification_and_value",
        "L2 diversification and bilateral trade value, 2017–2026 YTD", height=620)

display(concentration.assign(cr5_pct=100 * concentration["cr5"])[
    ["reference_year", "russia_flow", "effective_categories", "hhi_10000", "cr5_pct", "included_l2"]
].round({"effective_categories": 2, "hhi_10000": 0, "cr5_pct": 1}))

## 4. Prospective analysis: where does Russia stand in Vietnam's import market?

This section compares Russian bilateral supply with Vietnam's total imports for matching L2 categories. Market size, market dynamics, Russia's current share, and the change in that share are considered together. A low share alone does not imply an export opportunity: a final assessment requires evidence on competitiveness, logistics, regulation, and demand.


In [ ]:
vietnam_imports_raw = mapped_all[
    (mapped_all["fact_level"] == "national_commodity") & (mapped_all["series"] == "2N")
].copy()
vietnam_imports_l2 = (
    vietnam_imports_raw.groupby(["reference_year", "commodity_id", "canonical_name", "analytical_group_ru"],
                                as_index=False, dropna=False)["ytd_value_usd"].sum()
    .rename(columns={"ytd_value_usd": "vietnam_imports_usd"})
)
russia_supply_l2 = (
    l2[l2["series"] == "5N"]
    [["reference_year", "commodity_id", "canonical_name", "analytical_group_ru", "ytd_value_usd"]]
    .rename(columns={"ytd_value_usd": "imports_from_russia_usd"})
)
market_presence = russia_supply_l2.merge(
    vietnam_imports_l2,
    on=["reference_year", "commodity_id", "canonical_name", "analytical_group_ru"],
    how="inner", validate="one_to_one",
)
market_presence["russia_share"] = market_presence["imports_from_russia_usd"] / market_presence["vietnam_imports_usd"]
assert (market_presence["russia_share"] <= 1).all()

matched_total = market_presence.groupby("reference_year", as_index=False).agg(
    imports_from_russia_usd=("imports_from_russia_usd", "sum"),
    vietnam_imports_usd=("vietnam_imports_usd", "sum"),
    matched_l2=("commodity_id", "nunique"),
)
matched_total["russia_share"] = matched_total["imports_from_russia_usd"] / matched_total["vietnam_imports_usd"]

market_current = market_presence[
    (market_presence["reference_year"] == 2026) & (market_presence["commodity_id"] != "C074")
].copy()
market_current["vietnam_imports_usd_bn"] = market_current["vietnam_imports_usd"] / 1e9
market_current["russia_share_pct"] = market_current["russia_share"] * 100
market_current["imports_from_russia_usd_m"] = market_current["imports_from_russia_usd"] / 1e6
label_mask = (market_current["russia_share_pct"] >= 4) | (
    market_current["vietnam_imports_usd_bn"].between(15, 30)
)
market_current["label"] = np.where(label_mask, market_current["canonical_name"], "")

fig = px.scatter(
    market_current, x="vietnam_imports_usd_bn", y="russia_share_pct",
    size="imports_from_russia_usd_m", color="analytical_group_ru", text="label",
    color_discrete_map=L1_COLORS, size_max=55,
    labels={
        "vietnam_imports_usd_bn": "Vietnam import market, USD billion",
        "russia_share_pct": "Russia's supply share, %",
        "imports_from_russia_usd_m": "Russian exports, USD million",
        "analytical_group_ru": "L1",
    },
    hover_name="canonical_name",
    hover_data={"vietnam_imports_usd_bn": ":.2f", "russia_share_pct": ":.2f",
                "imports_from_russia_usd_m": ":.1f", "label": False},
)
fig.update_traces(textposition="top center", marker_line_color="white", marker_line_width=1)
present(fig, "11_market_presence", "Vietnam import market × Russia's share, 2026 YTD",
        height=720, right_margin=300, legend_orientation="v")

share_current = matched_total.loc[matched_total["reference_year"] == 2026, "russia_share"].iloc[0]
display(Markdown(
    f"Across matched L2 groups, Russia's aggregate share in January–September 2026 was **{share_current:.2%}**."
))

### 4.1. Potential-growth matrix: market CAGR × Russia's share

The horizontal axis shows CAGR of Vietnam's total L2 import market in 2017–2025; the vertical axis shows Russia's 2025 share; bubble size represents 2025 market size; and color represents L1. Tooltips retain absolute market change, annual-growth volatility, Russia's share, and Russian supply. Large growing markets with a small Russian share form a shortlist for further research, not an automatic export recommendation.

2026 YTD is intentionally excluded because an incomplete year would distort both CAGR and category positions in the matrix.


In [ ]:
market_cagr = build_cagr_metrics(
    vietnam_imports_l2[vietnam_imports_l2["commodity_id"] != "C074"],
    ["commodity_id", "canonical_name", "analytical_group_ru"],
    value_column="vietnam_imports_usd",
    low_base_usd=100_000_000,
)

market_2025 = market_presence[
    (market_presence["reference_year"] == CAGR_END_YEAR)
    & (market_presence["commodity_id"] != "C074")
][[
    "commodity_id", "canonical_name", "analytical_group_ru",
    "vietnam_imports_usd", "imports_from_russia_usd", "russia_share",
]].copy()
market_2025 = market_2025.rename(columns={
    "vietnam_imports_usd": "market_size_2025_usd",
    "imports_from_russia_usd": "russia_supply_2025_usd",
    "russia_share": "russia_share_2025",
})
market_share_2017 = market_presence[
    (market_presence["reference_year"] == CAGR_START_YEAR)
    & (market_presence["commodity_id"] != "C074")
][["commodity_id", "russia_share"]].rename(columns={"russia_share": "russia_share_2017"})

market_outlook = market_2025.merge(
    market_cagr.drop(columns=["canonical_name", "analytical_group_ru"]),
    on="commodity_id", how="left", validate="one_to_one",
).merge(market_share_2017, on="commodity_id", how="left", validate="one_to_one")
market_outlook["market_size_2025_usd_bn"] = market_outlook["market_size_2025_usd"] / 1e9
market_outlook["russia_supply_2025_usd_m"] = market_outlook["russia_supply_2025_usd"] / 1e6
market_outlook["russia_share_2025_pct"] = 100 * market_outlook["russia_share_2025"]
market_outlook["share_change_pp"] = 100 * (
    market_outlook["russia_share_2025"] - market_outlook["russia_share_2017"]
)

matrix_data = market_outlook[
    market_outlook["complete_period"] & market_outlook["cagr_pct"].notna()
].copy()
growth_candidates = matrix_data[
    (matrix_data["market_size_2025_usd_bn"] >= 0.5)
    & (matrix_data["value_2017_usd"] >= 100_000_000)
    & (matrix_data["cagr_pct"] > 0)
    & (matrix_data["russia_share_2025_pct"] < 2)
].nlargest(10, "market_size_2025_usd_bn")
candidate_ids = set(growth_candidates.head(6)["commodity_id"])
matrix_data["label"] = np.where(
    matrix_data["commodity_id"].isin(candidate_ids), matrix_data["canonical_name"], ""
)

fig = px.scatter(
    matrix_data,
    x="cagr_pct", y="russia_share_2025_pct", size="market_size_2025_usd_bn",
    color="analytical_group_ru", text="label", hover_name="canonical_name",
    color_discrete_map=L1_COLORS, size_max=62,
    labels={
        "cagr_pct": "Vietnam import-market CAGR, 2017–2025, %",
        "russia_share_2025_pct": "Russia's share in 2025, %",
        "market_size_2025_usd_bn": "2025 market size, USD billion",
        "analytical_group_ru": "L1",
        "absolute_change_usd": "Absolute market change, USD",
        "yoy_volatility_pp": "YoY volatility, pp",
        "share_change_pp": "Change in Russia's share, pp",
        "russia_supply_2025_usd_m": "Russian supply in 2025, USD million",
    },
    hover_data={
        "market_size_2025_usd_bn": ":.2f",
        "cagr_pct": ":+.1f",
        "russia_share_2025_pct": ":.2f",
        "absolute_change_usd": ":+,.0f",
        "yoy_volatility_pp": ":.1f",
        "share_change_pp": ":+.2f",
        "russia_supply_2025_usd_m": ":.1f",
        "label": False,
    },
)
fig.add_vline(x=0, line_dash="dot", line_color="#6F7780")
fig.add_hline(y=2, line_dash="dot", line_color="#AAB0B6")
fig.update_traces(textposition="top center", marker_line_color="white", marker_line_width=1)
present(
    fig, "12_market_growth_opportunities",
    "Vietnam import market: CAGR × Russia's share × market size, 2017–2025",
    height=760, right_margin=300, legend_orientation="v",
)

display(growth_candidates[[
    "canonical_name", "analytical_group_ru", "market_size_2025_usd_bn", "cagr_pct",
    "absolute_change_usd", "yoy_volatility_pp", "russia_share_2025_pct",
    "share_change_pp", "russia_supply_2025_usd_m",
]].rename(columns={
    "canonical_name": "L2", "analytical_group_ru": "L1",
    "market_size_2025_usd_bn": "2025 market, USD billion",
    "cagr_pct": "Market CAGR, %", "absolute_change_usd": "Absolute market change, USD",
    "yoy_volatility_pp": "YoY volatility, pp",
    "russia_share_2025_pct": "Russia's 2025 share, %",
    "share_change_pp": "Change in Russia's share, pp",
    "russia_supply_2025_usd_m": "Russian supply in 2025, USD million",
}).round(2))

## 5. Main findings and potential growth areas


In [ ]:
conc_2017 = concentration[concentration["reference_year"] == 2017].set_index("series")
conc_current = concentration[concentration["reference_year"] == 2026].set_index("series")
completed_last = annual_totals[annual_totals["reference_year"] == 2025].iloc[0]
current = annual_totals[annual_totals["reference_year"] == 2026].iloc[0]
top_export_driver = l2_change[l2_change["series"] == "5N"].nlargest(1, "delta_usd").iloc[0]
top_import_driver = l2_change[l2_change["series"] == "5X"].nlargest(1, "delta_usd").iloc[0]
turnover_cagr = total_cagr_summary.set_index("metric").loc["Trade turnover", "cagr_pct"]
candidate_names = ", ".join(growth_candidates["canonical_name"].head(3).tolist()) or "no categories meet the criteria"
display(Markdown(f"""
1. **Scale and growth:** trade turnover increased from **USD {start_total.turnover_usd/1e9:.2f} billion** in 2017 to **USD {completed_last.turnover_usd/1e9:.2f} billion** in 2025, a CAGR of **{turnover_cagr:.1f}% per year**. Turnover reached **USD {current.turnover_usd/1e9:.2f} billion** in January–September 2026, but this incomplete period is excluded from CAGR and is not used for a non-comparable YoY calculation.
2. **L1/L2 and stability:** CAGR is considered alongside absolute change, the number of growth years, and volatility. This separates a sustained trajectory from a low-base jump and prevents one endpoint from becoming a strategic conclusion.
3. **Commodity drivers:** `{top_export_driver.canonical_name}` made the largest positive contribution to the change in Russian exports from 2017 to 2026 YTD (**USD {top_export_driver.delta_usd/1e6:+.0f} million**), while `{top_import_driver.canonical_name}` led the change in Russian imports (**USD {top_import_driver.delta_usd/1e6:+.0f} million**). These are accounting contributions, not established causal effects.
4. **Diversification:** the effective number of L2 categories in Russian exports to Vietnam changed from **{conc_2017.loc['5N','effective_categories']:.1f} to {conc_current.loc['5N','effective_categories']:.1f}**, while Vietnamese exports to Russia changed from **{conc_2017.loc['5X','effective_categories']:.1f} to {conc_current.loc['5X','effective_categories']:.1f}**.
5. **Russia's position:** Russia's aggregate share across matched Vietnamese import L2 categories was **{share_current:.2%}** in 2026 YTD. The strategy matrix uses the 2025 share so that it matches the 2017–2025 market-CAGR horizon.
6. **Candidates for further research:** the formal screen—2025 market of at least USD 0.5 billion, 2017 base of at least USD 0.1 billion, positive CAGR, and Russia's share below 2%—identifies **{candidate_names}**. This is a research shortlist, not a ready-made export recommendation.
7. **Japan benchmark:** the dotted-line benchmark highlights differences in L1 specialization and supports hypotheses about the role of Russia's Far East. Claims about logistics advantages or supply redistribution require additional evidence.
"""))


## 6. Practical use and development of the analytical system

The notebook can evolve from a one-off statistical study into a recurring decision-support tool:

1. **Foreign-trade monitoring:** automatically refresh `country_total`, L1/L2 structure, CAGR, absolute change, YoY volatility, HHI, effective category count, and CR5 after each statistical release.
2. **Export-opportunity screening:** rank L2 categories by Vietnam import-market CAGR and scale, Russia's current share, absolute market growth, and annual-trajectory stability.
3. **Hypothesis testing:** add freight costs, routes through Russian Far Eastern ports, tariff and non-tariff restrictions, Russian production capacity, and competitor data, including Japan.
4. **Decision support:** maintain a monitored category list with thresholds, share-change alerts, and market cards for companies, chambers of commerce, and export institutions.
5. **Correct treatment of an incomplete year:** when monthly archives become available, compare 2026 YTD only with January–September 2025 and keep this measure separate from annual CAGR.

The next analytical step is not to expand the “promising” category list automatically, but to test each category for Russian supply availability, margins, logistics, and regulatory barriers.
